Author: Konstantin Malanchev and the LINCC Frameworks team

Last updated: November 2026

# Introduction to LSDB: Rubin DP2 color-color diagram

[LSDB](https://docs.lsdb.io) is a Python framework for working with large astronomical catalogs in the HATS format. In this notebook we'll show how to:
- Set up a Dask client
- Load the Rubin DP2 Object catalog (lazily) and select columns
- Understand HATS partitioning
- Select a region on the sky
- Select "bright" stars and plot a color-color diagram

> [!NOTE]
> This notebook uses Rubin DP2 data, so it can be run by Rubin data rights holders (DRHs) only, e.g. on the [Rubin Science Platform](https://data.lsst.cloud) with the latest weekly image.
> Since we would like to share the notebook and its outputs with everyone, we never print or plot individual rows of the DP2 catalog: all the outputs are either catalog metadata or aggregated statistics.

In [ ]:
# lsdb is pre-installed on RSP, uncomment the line below if you are on another platform
# %pip install -qq lsdb
# lsdb-rubin is not pre-installed on RSP
%pip install -qq lsdb-rubin

In [ ]:
# Standard library imports
import logging
import os
import warnings

# Third-party imports
import dask
import lsdb
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dask.distributed import Client
from lsdb_rubin import DP2FieldSearch
from matplotlib import rcParams
from matplotlib.colors import LogNorm

# Jupyter magic commands
%matplotlib inline

# Configuration
warnings.simplefilter("ignore")
logging.getLogger("distributed").setLevel(logging.WARNING)
rcParams["savefig.dpi"] = 150
rcParams["font.size"] = 14
plt.rc("font", family="serif")
mpl.rcParams["axes.linewidth"] = 1.5
dask.config.set({"dataframe.convert-string": False})

print(f"Version of lsdb is {lsdb.__version__}")

## Setup Dask cluster

LSDB uses the Dask framework for parallel and out-of-memory computing. Dask is a flexible library for parallel computing in Python that scales from single machines to large clusters. When using LSDB, it is usually worth setting up a Dask cluster (local or multi-node) to take advantage of parallel computing. With no Dask cluster, LSDB would use a single CPU core for the computations, which is prohibitive for large datasets.

The settings below are for a "Large" RSP container: sixteen workers, one per CPU, with one thread each. Our pipeline needs little memory, so we can afford a worker per CPU. By default, Dask splits the available memory between the workers, so each of them gets 2 GB of the 32 GB container. We also point Dask to `/deleted-sundays/`, a scratch directory where workers can spill data which doesn't fit into memory. As the name suggests, it is cleaned every Sunday.

On other facilities, use a scratch directory for `local_directory`, and use one worker per CPU, as long as each worker gets at least 2 GB of memory.

In [ ]:
client = Client(
    n_workers=16,
    threads_per_worker=1,
    local_directory=f"/deleted-sundays/{os.environ.get('USER', 'dask_scratch')}",
)

# Print the dashboard link and port
print(f"Dask is running at: {client.dashboard_link}")

## Open the DP2 Object catalog

The Rubin DP2 catalogs are available in HATS format on RSP under `/rubin/lsdb_data/dp2/`; please check [lsdb.io/dp2](https://lsdb.io/dp2) for the paths on other platforms. We'll use the Object catalog: static objects detected and measured on the coadded images.

Let's open it with `lsdb.open_catalog()`. We also specify only the columns we need: loading fewer columns is important for performance when working with large catalogs.

In [ ]:
# DP2 Object collection on RSP, check https://lsdb.io/dp2 for the paths on other platforms
DP2_OBJECT_PATH = "/rubin/lsdb_data/dp2/object_collection"

In [ ]:
object_catalog = lsdb.open_catalog(
    DP2_OBJECT_PATH,
    # coordinates and _healpix_29 are always loaded
    columns=[
        "objectId",
        "refExtendedness",
        "g_psfMag",
        "g_psfMagErr",
        "r_psfMag",
        "r_psfMagErr",
        "i_psfMag",
        "i_psfMagErr",
        "z_psfMag",
        "z_psfMagErr",
    ],
)
object_catalog

We've specified only the columns needed for our science case. This matters a lot here: the DP2 Object catalog has more than a thousand columns, and without an explicit column list LSDB would load a default subset of them, which is much slower and uses far more memory.

You can always inspect the full available column list with `catalog.all_columns`. Let's look at the columns related to the PSF photometry in the g band. Magnitude columns are not a part of the original Rubin data products, they were added to the HATS catalogs for convenience, see the [DP2 page on data.lsdb.io](https://data.lsdb.io/Rubin/DP2/object_collection) for details. Descriptions of the original columns are in the [DP2 schema browser](https://sdm-schemas.lsst.io/dp2.html).

In [ ]:
print(f"Total number of columns: {len(object_catalog.all_columns)}")
[column for column in object_catalog.all_columns if column.startswith("g_psf")]

## Lazy Operations

When working with large datasets, there is too much data to be loaded into memory at once. To get around this, LSDB uses the HATS format which partitions a catalog into HEALPix tiles and works on one partition at a time. This also allows the computation to be parallelized to work on multiple partitions at once.

For this reason, operations in LSDB are performed 'lazily'. This means when a catalog is read using `open_catalog`, the actual catalog data isn't being read from storage. Instead, it only loads the metadata such as the column schema and the HEALPix structure of the partitions. When an operation like a cone search or a row filter is called on a catalog, the data is not actually loaded and operated on when the line is executed. Instead, the operation is added to a task graph, which is executed only when the result is requested.

![lazy loading diagram](https://raw.githubusercontent.com/astronomy-commons/lsdb/main/docs/_static/lazy_diagram.svg)

So when we look at a catalog that has been lazy loaded we see the DataFrame without the data, just the columns and the number of partitions (including the HATS index of each partition encoding which HEALPix cell the partition is in). This is also why displaying a lazy catalog is safe from the data rights point of view: it shows the schema, not the data.

## HATS (Hierarchical Adaptive Tiling Scheme) Partitioning

To make parallel computation efficient, HATS partitions contain roughly the same number of rows. This is achieved by using different HEALPix pixel sizes across the sky depending on source density; denser regions of the sky get smaller pixels. We can visualise the pixel layout of our catalog with `plot_pixels()`.

In [ ]:
fig = plt.figure(figsize=(20, 10))
object_catalog.plot_pixels(fig=fig)
plt.title("Rubin DP2 Object HATS partition map")
plt.show()

## Select a region on the sky

DP2 covers about 3000 square degrees of the sky. The `lsdb-rubin` package ([GitHub](https://github.com/astronomy-commons/lsdb-rubin)) knows the centers of the DP2 fields, and its `DP2FieldSearch` is a cone search around one of them. Let's select a 1-degree cone around the center of the Extended Chandra Deep Field South (ECDFS), one of the Rubin deep drilling fields. We apply it with `.search()`, which is lazy as well: only the partitions overlapping the cone would be read when we compute the result.

In [ ]:
print(", ".join(DP2FieldSearch.FIELDS))

# 1-degree cone around the ECDFS deep drilling field
field_search = DP2FieldSearch("DDF ECDFS", radius_arcsec=1.0 * 3600)

object_cone = object_catalog.search(field_search)
object_cone

## Select "bright" stars

We want objects which are likely stars, with well-measured colors. We select:
- point-like objects with `refExtendedness == 0`, the extendedness measured in the reference band, the band which was used for the detection and shape measurements of the object;
- objects with PSF magnitude errors smaller than 0.05 mag in all of g, r, i, z bands: we want small errors, which also selects brighter objects.

`query()` is lazy as well, so nothing is computed yet.

In [ ]:
BANDS = "griz"
mag_err_query = " and ".join(f"{band}_psfMagErr < 0.05" for band in BANDS)
print(mag_err_query)

stars = object_cone.query(f"refExtendedness == 0 and {mag_err_query}")
stars

## Color-color diagram

For a small region we could call `stars.compute()`, get a pandas DataFrame and plot every object. But a scatter plot of individual DP2 objects is exactly what we don't want to share with non-DRHs. Instead, we compute an aggregated product, a 2D histogram of the g−r and i−z colors, and plot it.

As a bonus, this approach scales to the whole catalog: we compute a partial histogram for each partition with `map_partitions()` and sum them up at the end. Note that all partial histograms must use the same bins, so we fix the bin edges in advance.

In [ ]:
GR_BINS = np.linspace(-0.5, 2.0, 126)
IZ_BINS = np.linspace(-0.5, 1.0, 76)


def color_color_hist(df):
    """Partial 2D histogram of g-r vs i-z for a single partition.

    Returns non-empty bins only, as a "long" table of bin indices and counts.
    """
    counts, _, _ = np.histogram2d(
        df["g_psfMag"] - df["r_psfMag"],
        df["i_psfMag"] - df["z_psfMag"],
        bins=[GR_BINS, IZ_BINS],
    )
    gr_idx, iz_idx = np.nonzero(counts)
    return pd.DataFrame(
        {"gr_bin": gr_idx, "iz_bin": iz_idx, "count": counts[gr_idx, iz_idx].astype(np.int64)}
    )


# Dask needs to know the output schema in advance
meta = {"gr_bin": int, "iz_bin": int, "count": int}
partial_hists = stars.map_partitions(color_color_hist, meta=meta)
partial_hists

Now we call `compute()`, which loads the data, applies all the operations from the task graph, and returns the result: here it is a table of bin counts, one per partition, not the DP2 rows. Since `map_partitions()` created a new data frame, it has no data-rights protected columns anymore: no coordinates, no `_healpix_29` index, just bin indices and counts.

In [ ]:
partial_hists_df = partial_hists.compute()

# Sum the partial histograms
hist = np.zeros((len(GR_BINS) - 1, len(IZ_BINS) - 1), dtype=np.int64)
np.add.at(hist, (partial_hists_df["gr_bin"], partial_hists_df["iz_bin"]), partial_hists_df["count"])

# A bin with a single object exposes its colors up to the bin size, so we remove such bins
hist[hist == 1] = 0
print(f"Total number in the histogram: {hist.sum():,}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
mesh = ax.pcolormesh(IZ_BINS, GR_BINS, hist, norm=LogNorm(), cmap="viridis")
fig.colorbar(mesh, ax=ax, label="Number")
ax.set_xlabel("$i - z$ [mag]")
ax.set_ylabel("$g - r$ [mag]")
ax.set_title("Rubin DP2 point sources around ECDFS")
plt.tight_layout()
plt.show()

We expect to see the stellar locus: the narrow sequence along which most of the stars lie.

In [ ]:
client.close()

## Problem: Palomar 12 color-magnitude diagram

**Your task:** plot a color-magnitude diagram of the globular cluster Palomar 12.

**Steps:**
1. Select a small cone around the cluster with `lsdb.ConeSearch(ra=326.66, dec=-21.25, radius_arcsec=0.5 * 3600)`.
2. Select point sources, as we did above. You need only r and z bands this time, so you can load fewer columns.
3. Plot an r vs r−z color-magnitude diagram. Adapt `color_color_hist()` from above to compute a 2D histogram, and remove the bins with a single object.

One possible solution is in `Notebook_1b_ColorMagnitude.ipynb`.